---
title: Estimating allelic trajectories from PooledSeq
description: Using SDEs to estimate allelic trajectories and associated evolutionary parameters from PooledSeq
categories:
  - Methods & Inference
date: Jan-2021
image: cover_images/BayesEvolveResequence.png
cap-location: margin
title-block-banner: cover_images/BayesEvolveResequence_trajectories_small.png
jupyter:
  jupytext:
    formats: qmd:quarto,ipynb
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.5
  kernelspec:
    display_name: Python 3 (main)
    language: python
    name: main
---


An evolve-and-resequence experiment gives you allele frequencies moving in real time: expose a population to a selective pressure, let it respond for a number of generations, and sequence the pooled population as a whole, losing individual haplotypes, at several time points. The design trades individual genotypes for throughput, allowing entire replicate populations to be tracked at a fraction of the cost, and it has deservedly become the workhorse of adaptive evolution studies. The standard analysis, though, was inherited from contingency-table thinking, and the mismatch between the biology and the test shows up in the error rates.

The classical workflow is built on the Cochran-Mantel-Haenszel (CMH) test: an association test between treatment and time over the read counts, stratified across replicates, with a Breslow-Day test to check that the strata agree. Easy to run but biologically indefensible. The pairing between a specific control replicate and a specific treatment replicate is typically an artifact of the experimental layout rather than a real connection; intermediate time points added after the fact don't fit the strata; and, most importantly, the test treats allele-frequency changes as deterministic responses to treatment, ignoring the stochastic nature of the underlying population-genetic process. The consequence is an inflated false-positive rate: drift alone can produce frequency changes that look like selection, and the CMH framework has no vocabulary for that distinction.

## A model of the experiment actually run

The fix is to embed the inference in an explicit mechanistic model of the population-genetic dynamics. The expected change in the frequency \(p\) of a focal allele under selection is

$$
\Delta p = \frac{p\big(W_{11}-W_{01}\big)+(1-p)\big(W_{01}-W_{00}\big)}{\bar{W}}\,p(1-p),
$$

where \(W_{11}=(1+s)^2\), \(W_{00}=1\), and \(W_{01}=(1+sh)\) are the fitnesses of the two homozygotes and the heterozygote, respectively. The model is thus not restricted to additive architectures: arbitrary dominance (\(h\)) and selection coefficients (\(s\)) are part of the vocabulary.

Selection alone is not the whole story in a pooled-sequencing experiment, because two additional binomial processes act on the observed frequencies. Genetic drift operates during the discrete, fixed-size Wright-Fisher generations, and sequencing itself subsamples individuals at a given depth \(d\). Both effects can be modeled as binomial draws, which allows an explicit variance term to be carried through to downstream inference instead of being treated as nuisance noise.

![Estimated allele-frequency trajectories over the generations of the experiment, each color family a different variant; the fan of trajectories reflects the uncertainty the model carries.](cover_images/BayesEvolveResequence_trajectories.png){#fig-traj}

What the model estimates is best seen in the trajectories (@fig-traj): variants responding to the selection pressure climb within their color family, while the spread of the draws around each trajectory is the posterior doing its job: carrying the uncertainty that drift and sequencing depth actually generate.

The payoff is a framework that separates the signal from the stochastic background by construction, rather than by post-hoc correction. It captures the influence of dominance and selection on the allele-frequency trajectories; it provides a realistic variance estimate that accounts for both biological drift and technical sampling; and it extends to more than two time points or to hierarchical experimental designs, situations where the stratified CMH analysis would require uncomfortable contortions.

## Take-home messages

- Frequencies from pooled sequencing are noisy, renormalized samples of a stochastic process; a test with no vocabulary for drift reports drift as selection.
- The replicate pairing the CMH test requires encodes bench layout; biology never enters it.
- Modeling the experiment you actually ran (Wright-Fisher drift, sequencing depth, dominance) is cheaper than defending the mismatch, and extends naturally to extra time points and hierarchical designs.

Grounding the statistical test in a biologically plausible stochastic model yields more reliable estimates of the evolutionary parameters, a step toward making the inference layer of pooled-sequencing experiments as sophisticated as the sequencing layer itself.